In [3]:
import json
import pandas as pd

# load distances
with open("algeria_distances.json", "r") as f:
    distances = json.load(f)

# load CSVs
employees_file = pd.read_csv(r"data\emplo.csv")
jobs_file = pd.read_csv(r"data\jobs.csv")


# Node class

### done

In [4]:
import queue
from copy import deepcopy
class Node:
    def __init__(self, state, parent=None, action=None, g=0, f=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.g = g  
        self.f = f  
        if parent is None:
            self.depth = 0
        else:
            self.depth = parent.depth + 1

    def __hash__(self):
        if isinstance(self.state, list):
            state_tuple = tuple([tuple(row) for row in self.state])
            return hash(state_tuple)
        return hash(self.state)

    def __eq__(self, other):
        return isinstance(other, Node) and self.state == other.state

    def __gt__(self, other):
        return isinstance(other, Node) and self.f > other.f
    
    # Add lt for proper priority queue comparison
    def __lt__(self, other):
        return isinstance(other, Node) and self.f < other.f



### Similary Function :


done

In [5]:
import csv
import math

# Map common degree strings to numeric levels
EDU_LEVELS = {
    'bac': 12,
    'licence': 15,
    'master': 17,
    'doctorat': 20,
    "ingénieur d'état": 18
}

# Core compatibility scoring function
# weights must sum to 1.0

# Updated weights (sum to exactly 1.0)
WEIGHTS = {
    'skills': 0.20,         # Technical capabilities
    'experience': 0.20,     # Relevant experience
    'contract_type': 0.15,  # Employment contract match
    'education': 0.12,      # Educational qualifications
    'salary': 0.10,         # Salary expectations
    'language': 0.10,       # Language proficiency (new component)
    'sector': 0.08,         # Industry alignment
    'location': 0.05        # Geographic proximity
}

def parse_languages(lang_str):
    """Parse language strings handling Algerian context inconsistencies"""
    return set([lang.strip().lower() 
              for lang in str(lang_str).split(',') 
              if lang.strip()])

def noncompatibility(job, employee, salary_multiplier=2, weights=WEIGHTS, distances=None, return_components=False):
    # Validate weights
    if abs(sum(weights.values()) - 1.0) > 1e-6:
        raise ValueError("Weights must sum to 1.0")

    # Language proficiency scoring
    job_langs = parse_languages(job.get('language_proficiency', ''))
    emp_langs = parse_languages(employee.get('language_proficiency', ''))
    
    if not job_langs:
        lang_score = 0.0
    else:
        matched = len(job_langs & emp_langs)
        lang_score = 1 - (matched / len(job_langs))
    
    # Skills match
    job_skills = set(s.strip().lower() for s in str(job.get('inferred_skills', '')).split(','))
    emp_skills = set(s.strip().lower() for s in str(employee.get('technical_skills', '')).split(','))
    skills_score = 0.0 if not job_skills else 1 - len(job_skills & emp_skills)/len(job_skills)

    # Experience score
    emp_exp = employee.get('years_experience', 0)
    min_exp = job.get('experience_min_req', 0)
    max_exp = job.get('experience_max_req', 0)
    if emp_exp < min_exp:
        exp_score = 1 - 0.9*emp_exp/min_exp if min_exp else 1.0
    elif emp_exp < max_exp:
        exp_score = 0.1 - 0.1*(emp_exp-min_exp)/(max_exp-min_exp) if max_exp != min_exp else 0.0
    else:
        exp_score = 0.0

    # Contract type
    contract_score = float(str(employee.get('contract_type', '')).lower() != str(job.get('contract_type', '')).lower())

    # Education
    emp_edu = employee.get('edu_value', 0)
    job_edu = job.get('education_years', job.get('edu_value', 0))
    if emp_edu < job_edu:
        edu_score = 1.0
    elif emp_edu < 20:
        edu_score = 0.1 - 0.1*(emp_edu-job_edu)/(20-job_edu) if job_edu != 20 else 0.0
    else:
        edu_score = 0.0

    # Salary
    E = employee.get('salary', 0)
    O = job.get('predicted_salary_dzd', 0)
    if not O:
        salary_score = 0.0 if E <= 0 else 1.0
    else:
        if E <= O:
            salary_score = 0.0
        elif E >= salary_multiplier * O:
            salary_score = 1.0
        else:
            salary_score = (E - O) / ((salary_multiplier - 1) * O)

    # Sector
    sector_score = float(str(employee.get('sector', '')).lower() != str(job.get('sector', '')).lower())

    # Location
    loc_score = 1.0
    if distances and job.get('job location') and employee.get('city'):
        dist = distances.get(employee['city'], {}).get(job['job location'], float('inf'))
        loc_score = min(dist / 2500, 1.0) if dist != float('inf') else 1.0

    # Calculate total score
    total_score = (
        weights['skills'] * skills_score +
        weights['experience'] * exp_score +
        weights['contract_type'] * contract_score +
        weights['education'] * edu_score +
        weights['salary'] * salary_score +
        weights['language'] * lang_score +
        weights['sector'] * sector_score +
        weights['location'] * loc_score
    )

    if return_components:
        return total_score, {
            'skills': skills_score,
            'experience': exp_score,
            'contract_type': contract_score,
            'education': edu_score,
            'salary': salary_score,
            'language': lang_score,
            'sector': sector_score,
            'location': loc_score
        }
    return total_score

def parse_skills(skill_str):
    """Handle Algerian-specific skill formatting"""
    return set([skill.strip().lower().replace('billingual', 'bilingual') 
              for skill in str(skill_str).split(',')])
    
    
def depth_specific_score(job, employee, depth, salary_multiplier=2, weights=WEIGHTS, distances=None):
    """
    Calculate compatibility score up to a specific depth level, where each depth corresponds to an attribute cluster.
    
    Args:
        job: Job data dictionary
        employee: Employee data dictionary
        depth: Current depth in the search tree (determines which attributes to include)
        salary_multiplier: Multiplier for salary comparison
        weights: Dictionary of attribute weights
        distances: Distance matrix for location scoring
    
    Returns:
        Partial compatibility score up to the specified depth
    """
    # Define the order of attributes by depth
    ATTRIBUTE_ORDER = [
        'skills',        # Depth 0
        'experience',    # Depth 1
        'contract_type', # Depth 2
        'education',     # Depth 3
        'salary',        # Depth 4
        'language',      # Depth 5
        'sector',        # Depth 6
        'location'       # Depth 7
    ]
    
    # Validate depth
    if depth >= len(ATTRIBUTE_ORDER):
        depth = len(ATTRIBUTE_ORDER) - 1
    
    total_score = 0.0
    
    # Calculate each component up to the current depth
    for current_depth in range(depth + 1):
        attribute = ATTRIBUTE_ORDER[current_depth]
        
        if attribute == 'skills':
            # Skills match
            job_skills = set(s.strip().lower() for s in str(job.get('inferred_skills', '')).split(','))
            emp_skills = set(s.strip().lower() for s in str(employee.get('technical_skills', '')).split(','))
            component_score = 0.0 if not job_skills else 1 - len(job_skills & emp_skills)/len(job_skills)
            
        elif attribute == 'experience':
            # Experience score
            emp_exp = employee.get('years_experience', 0)
            min_exp = job.get('experience_min_req', 0)
            max_exp = job.get('experience_max_req', 0)
            if emp_exp < min_exp:
                component_score = 1 - 0.9*emp_exp/min_exp if min_exp else 1.0
            elif emp_exp < max_exp:
                component_score = 0.1 - 0.1*(emp_exp-min_exp)/(max_exp-min_exp) if max_exp != min_exp else 0.0
            else:
                component_score = 0.0
                
        elif attribute == 'contract_type':
            # Contract type
            component_score = float(str(employee.get('contract_type', '')).lower() != str(job.get('contract_type', '')).lower())
            
        elif attribute == 'education':
            # Education
            emp_edu = employee.get('edu_value', 0)
            job_edu = job.get('education_years', job.get('edu_value', 0))
            if emp_edu < job_edu:
                component_score = 1.0
            elif emp_edu < 20:
                component_score = 0.1 - 0.1*(emp_edu-job_edu)/(20-job_edu) if job_edu != 20 else 0.0
            else:
                component_score = 0.0
                
        elif attribute == 'salary':
            # Salary
            E = employee.get('salary', 0)
            O = job.get('predicted_salary_dzd', 0)
            if not O:
                component_score = 0.0 if E <= 0 else 1.0
            else:
                if E <= O:
                    component_score = 0.0
                elif E >= salary_multiplier * O:
                    component_score = 1.0
                else:
                    component_score = (E - O) / ((salary_multiplier - 1) * O)
                    
        elif attribute == 'language':
            # Language proficiency
            job_langs = parse_languages(job.get('language_proficiency', ''))
            emp_langs = parse_languages(employee.get('language_proficiency', ''))
            if not job_langs:
                component_score = 0.0
            else:
                matched = len(job_langs & emp_langs)
                component_score = 1 - (matched / len(job_langs))
                
        elif attribute == 'sector':
            # Sector
            component_score = float(str(employee.get('sector', '')).lower() != str(job.get('sector', '')).lower())
            
        elif attribute == 'location':
            # Location
            component_score = 1.0
            if distances and job.get('job location') and employee.get('city'):
                dist = distances.get(employee['city'], {}).get(job['job location'], float('inf'))
                component_score = min(dist / 2500, 1.0) if dist != float('inf') else 1.0
        
        # Add weighted component to total score
        total_score += weights[attribute] * component_score
    
    return total_score

### this part is for the old formulation

In [6]:

def compute_top_matches(job_seekers_file, jobs_file, distances=None, output_file='scores.csv'):
    """
    Read seekers and jobs from CSVs, score and return top 10 per job.
    Columns mapped inside to match real CSV headers.
    """
    seekers = []
    with open(job_seekers_file, encoding='utf-8-sig') as f:
        for row in csv.DictReader(f):
            # convert edu level string to number if needed
            try:
                edu_val = int(row['edu_value'])
            except ValueError:
                edu_val = EDU_LEVELS.get(row['highest_education'].strip().lower(), 0)

            seekers.append({
                'technical_skills': row['technical_skills'],
                'years_experience': int(row['years_experience']),
                'salary': float(row.get('salary', row.get('predicted_salary_dzd', 0))),
                'city': row['city'],
                'job_interest': row.get('job_interest', ''),
                'sector': row['sector'],
                'edu_value': edu_val,
                'job_id': row.get('first_name', '')  # or use another unique ID
            })

    jobs = []
    with open(jobs_file, encoding='utf-8-sig') as f:
        for row in csv.DictReader(f):
            try:
                job_edu_val = int(row['education_years'])
            except ValueError:
                job_edu_val = EDU_LEVELS.get(row['education requirement'].strip().lower(), 0)

            jobs.append({
                'inferred_skills': row['inferred_skills'],
                'experience_min_req': int(row['experience_min_req']),
                'experience_max_req': int(row['experience_max_req']),
                'predicted_salary_dzd': float(row['predicted_salary_dzd']),
                'job location': row['job location'],
                'job_interest': row.get('job_interest', ''),
                'sector': row['sector'],
                'education_years': job_edu_val,
                'edu_value': job_edu_val,
                'job_id': row.get('job title', '')
            })

    scores = []
    for job in jobs:
        job_scores = []
        for seeker in seekers:
            sc = noncompatibility(job, seeker, distances=distances)
            job_scores.append((seeker['job_id'], sc))
        top10 = sorted(job_scores, key=lambda x: x[1])[:10]
        scores.append({'job_id': job['job_id'], 'top_10': top10})

    with open(output_file, 'w', newline='', encoding='utf-8') as outf:
        writer = csv.writer(outf)
        writer.writerow(['job_id', 'rank', 'seeker_id', 'score'])
        for entry in scores:
            for rank, (sid, sc) in enumerate(entry['top_10'], 1):
                writer.writerow([entry['job_id'], rank, sid, f"{sc:.4f}"])

    return scores


# make a problem definition where you make all the standard methods like expand node..

we need the following in problem formulation class:
is_goal()
initial_state
expand_node()


In [7]:
class Job_matching:
    def __init__(self, initial_state, goal_test, state_transition_model, actions):
        self.initial_state = initial_state
        self.goal_test = goal_test
        self.state_transition_model = state_transition_model
        self.actions = actions
        # Precompute job-employee scores for quick lookup
        self.job_emp_scores = {}
        for job_id, emp_list in self.state_transition_model.items():
            for emp_info in emp_list:
                emp_id = emp_info[0]
                score = emp_info[1]
                self.job_emp_scores[(job_id, emp_id)] = score

    def is_goal(self, state):
        # Check if all jobs are assigned
        assigned_jobs = {job_id for (job_id, emp_id) in state}
        all_jobs = set(self.state_transition_model.keys())
        if assigned_jobs != all_jobs:
            return False
        # Check if total score meets goal_test (for local search)
        if self.goal_test > 0:
            total_score = sum(self.job_emp_scores.get((job_id, emp_id), 0) for (job_id, emp_id) in state)
            return total_score <= self.goal_test
        return True  # Global search: all jobs assigned

    def expand_node(self, node):
        current_state = node.state
        assigned_employees = {emp_id for (job_id, emp_id) in current_state}
        assigned_jobs = {job_id for (job_id, emp_id) in current_state}
        all_jobs = set(self.state_transition_model.keys())
        remaining_jobs = [job_id for job_id in all_jobs if job_id not in assigned_jobs]
        
        if not remaining_jobs:
            return []
        
        next_job = remaining_jobs[0]  # Process jobs in transition model order
        child_nodes = []
        
        for emp_info in self.state_transition_model[next_job]:
            emp_id, emp_score = emp_info[0], emp_info[1]
            if emp_id in assigned_employees:
                continue
            
            new_state = list(current_state)
            new_state.append((next_job, emp_id))
            new_g = node.g + emp_score
            
            # Calculate heuristic for remaining jobs
            remaining_jobs_after = [job for job in remaining_jobs if job != next_job]
            sum_heuristic = 0
            assigned_in_child = assigned_employees.copy()
            assigned_in_child.add(emp_id)
            
            for job in remaining_jobs_after:
                for emp_candidate, score in self.state_transition_model[job]:
                    if emp_candidate not in assigned_in_child:
                        sum_heuristic += score
                        break
            
            new_f = new_g + sum_heuristic
            child_node = Node(new_state, node, emp_id, new_g, new_f)
            child_nodes.append(child_node)
        
        return child_nodes

#  General Search algorithms:

In [8]:
class GeneralSearch:
    def __init__(self, problem):
        self.problem = problem
        self.use_cost = False
        self.use_heuristic = True

    def set_frontier(self, search_strategy="A*"):
        frontier = queue.PriorityQueue()
        self.use_heuristic = True
        if search_strategy == "A*":
            self.use_cost = True     
        elif search_strategy == "Greedy":
            self.use_cost = False
        else:
            raise ValueError("Unsupported search strategy: " + str(search_strategy))
        return frontier

    def search(self, search_strategy="A*", max_depth=float('inf')):
        frontier = self.set_frontier(search_strategy)
        explored = set()
        initial_node = Node(self.problem.initial_state)
        frontier.put(initial_node)

        while not frontier.empty():
            node = frontier.get()
            if self.problem.is_goal(node.state):
                print("Goal reached!")
                return node
            
            # Convert list state to tuple for hashing
            state_tuple = tuple(tuple(pair) for pair in node.state)
            
            if node.depth > max_depth or state_tuple in explored:
                continue
                
            explored.add(state_tuple)
            child_nodes = self.problem.expand_node(node)
            
            for child_node in child_nodes:
                # Convert child state to tuple for comparison
                child_state_tuple = tuple(tuple(pair) for pair in child_node.state)
                if child_state_tuple not in explored:
                    frontier.put(child_node)
                    
        return None
    

In [9]:
import time
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Callable, Any
from collections import defaultdict

def evaluate_search_algorithms(
    seekers_data: List[Dict],
    jobs_data: List[Dict],
    algorithm_functions: Dict[str, Callable],
    distance_matrix: Dict[str, Dict[str, float]] = None,
    sample_size: int = None,
    weights: Dict[str, float] = None,
    matching_type: str = "job_to_seekers"  # or "seeker_to_jobs"
) -> Dict[str, Any]:
    """
    Evaluate multiple search algorithms for one-to-many job matching.
    
    Args:
        seekers_data: List of job seeker dictionaries
        jobs_data: List of job dictionaries
        algorithm_functions: Dictionary of algorithm functions to test
        distance_matrix: Optional distance matrix for location scoring
        sample_size: Number of seekers/jobs to use (None for all)
        weights: Dictionary of weights for scoring
        matching_type: "job_to_seekers" (one job, many employees) or 
                     "seeker_to_jobs" (one employee, many jobs)
        
    Returns:
        Dictionary containing:
        - results: DataFrame with metrics for each algorithm
        - plots: Dictionary of matplotlib figures
        - raw_data: Detailed results from each algorithm
    """
    
    # Set default weights if not provided
    if weights is None:
        weights = {
            'skills': 0.20,
            'experience': 0.20,
            'contract_type': 0.15,
            'education': 0.12,
            'salary': 0.10,
            'language': 0.10,
            'sector': 0.08,
            'location': 0.05
        }
    
    # Sample data if requested
    if sample_size is not None:
        seekers_data = seekers_data[:sample_size]
        jobs_data = jobs_data[:sample_size]
    
    results = []
    raw_data = {}
    plots = {}
    
    for algo_name, algo_func in algorithm_functions.items():
        print(f"\nRunning {algo_name} algorithm ({matching_type})...")
        
        start_time = time.time()
        
        try:
            # Run algorithm with timing
            algo_result = algo_func(
                seekers_data.copy(),
                jobs_data.copy(),
                distances=distance_matrix,
                weights=weights,
                matching_type=matching_type
            )
            
            run_time = time.time() - start_time
            
            # Calculate metrics
            metrics = calculate_metrics(
                algo_result, 
                seekers_data, 
                jobs_data, 
                weights,
                matching_type
            )
            metrics['time_seconds'] = run_time
            
            results.append({
                'algorithm': algo_name,
                **metrics
            })
            
            raw_data[algo_name] = {
                'assignments': algo_result['assignments'],
                'scores': algo_result['scores'],
                'time': run_time
            }
            
            print(f"{algo_name} completed in {run_time:.2f} seconds")
            print(f"Average score: {metrics['avg_score']:.3f}")
            
        except Exception as e:
            print(f"Error running {algo_name}: {str(e)}")
            results.append({
                'algorithm': algo_name,
                'error': str(e)
            })
    
    # Create results DataFrame
    results_df = pd.DataFrame(results)
    
    # Generate comparison plots
    plots['score_time_comparison'] = plot_score_time_comparison(results_df)
    plots['metric_comparison'] = plot_metric_comparison(results_df)
    plots['assignment_distribution'] = plot_assignment_distribution(raw_data, matching_type)
    
    return {
        'results': results_df,
        'plots': plots,
        'raw_data': raw_data
    }

def calculate_metrics(
    algo_result: Dict,
    seekers_data: List[Dict],
    jobs_data: List[Dict],
    weights: Dict[str, float],
    matching_type: str
) -> Dict[str, float]:
    """
    Calculate evaluation metrics for algorithm results for one-to-many matching.
    """
    assignments = algo_result['assignments']
    scores = algo_result['scores']
    
    # Count assignments per job/employee
    if matching_type == "job_to_seekers":
        assignment_counts = defaultdict(int)
        for job_id, _ in assignments:
            assignment_counts[job_id] += 1
        avg_assignments = np.mean(list(assignment_counts.values())) if assignment_counts else 0
    else:  # seeker_to_jobs
        assignment_counts = defaultdict(int)
        for _, seeker_id in assignments:
            assignment_counts[seeker_id] += 1
        avg_assignments = np.mean(list(assignment_counts.values())) if assignment_counts else 0
    
    # Basic metrics
    metrics = {
        'avg_score': np.mean(list(scores.values())),
        'min_score': np.min(list(scores.values())),
        'max_score': np.max(list(scores.values())),
        'std_score': np.std(list(scores.values())),
        'avg_assignments': avg_assignments,
        'unmatched_jobs': len(jobs_data) - len({a[0] for a in assignments}) if matching_type == "job_to_seekers" else 0,
        'unmatched_seekers': len(seekers_data) - len({a[1] for a in assignments}) if matching_type == "seeker_to_jobs" else 0
    }
    
    # Calculate component scores
    component_scores = {k: [] for k in weights.keys()}
    
    for job_id, seeker_id in assignments:
        job = next(j for j in jobs_data if j['job_id'] == job_id)
        seeker = next(s for s in seekers_data if s['seeker_id'] == seeker_id)
        
        # Get component scores
        _, components = noncompatibility(job, seeker, weights=weights, return_components=True)
        
        for k, v in components.items():
            component_scores[k].append(v)
    
    # Add average component scores
    for k, v in component_scores.items():
        metrics[f'avg_{k}_score'] = np.mean(v) if v else 0
    
    return metrics

def plot_assignment_distribution(raw_data: Dict[str, Any], matching_type: str) -> plt.Figure:
    """Plot distribution of assignments per job/employee."""
    fig, axes = plt.subplots(len(raw_data), 1, figsize=(10, 5*len(raw_data)))
    
    if len(raw_data) == 1:
        axes = [axes]
    
    for (algo_name, algo_data), ax in zip(raw_data.items(), axes):
        assignments = algo_data['assignments']
        
        if matching_type == "job_to_seekers":
            # Count seekers per job
            job_counts = defaultdict(int)
            for job_id, _ in assignments:
                job_counts[job_id] += 1
            counts = list(job_counts.values())
            title = f"{algo_name}: Seekers per Job Distribution"
            xlabel = "Number of Seekers per Job"
        else:
            # Count jobs per seeker
            seeker_counts = defaultdict(int)
            for _, seeker_id in assignments:
                seeker_counts[seeker_id] += 1
            counts = list(seeker_counts.values())
            title = f"{algo_name}: Jobs per Seeker Distribution"
            xlabel = "Number of Jobs per Seeker"
        
        ax.hist(counts, bins=range(0, max(counts)+2 if counts else [0], align='left', rwidth=0.8))
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Frequency")
        ax.grid(True)
    
    plt.tight_layout()
    return fig

# Example algorithm wrappers with one-to-many support
def run_a_star(seekers, jobs, distances=None, weights=None, matching_type="job_to_seekers"):
    """Wrapper for A* algorithm with one-to-many support."""
    # Modify your A* implementation to handle:
    # - For job_to_seekers: Find best N seekers for each job
    # - For seeker_to_jobs: Find best N jobs for each seeker
    pass

def run_greedy(seekers, jobs, distances=None, weights=None, matching_type="job_to_seekers"):
    """Wrapper for Greedy algorithm with one-to-many support."""
    pass

def run_genetic(seekers, jobs, distances=None, weights=None, matching_type="job_to_seekers"):
    """Wrapper for Genetic Algorithm with one-to-many support."""
    # Modify chromosome representation to handle one-to-many
    pass

def run_csp(seekers, jobs, distances=None, weights=None, matching_type="job_to_seekers"):
    """Wrapper for CSP solver with one-to-many support."""
    pass
def main():
    try:
        # Compute top matches using pre-loaded data
        scores = compute_top_matches(
            'emplo_final.csv',  # job seekers CSV filename
            'jobs_final.csv',    # jobs CSV filename
            distances_dict,
            'scores.csv'
        )

        # Build state transition model
        state_transition_model = {
            entry['job_id']: [(emp_id, score) for emp_id, score in entry['top_10']]
            for entry in scores
        }

        # Initialize problem
        problem = Job_matching(
            initial_state=[],
            goal_test=0,
            state_transition_model=state_transition_model,
            actions=None
        )

        # Execute search
        search = GeneralSearch(problem)
        solution_node = search.search(search_strategy="A*")

        # Display results
        if solution_node:
            print("\n=== Optimal Assignments ===")
            total_score = 0
            for job_id, emp_id in solution_node.state:
                score = problem.job_emp_scores[(job_id, emp_id)]
                total_score += score

                print(f"\nJob: {job_id}")
                print(f"  Employee: {emp_id}")
                print(f"  Match Score: {score:.4f}")
                print("─" * 50)

            print(f"\nTotal Compatibility Score: {total_score:.4f}")
        else:
            print("No valid assignment found.")

    except FileNotFoundError as e:
        print(f"Error: Missing required file - {e.filename}")
    except Exception as e:
        print(f"An error occurred: {str(e)}")
